# RK1828 YOLO 目标检测 Demo（YOLOv5/v6/v8 + YOLO26）

一次跑通 6 个 YOLO 模型，理解两类输出格式：

| 模型 | 输出 | 后处理 |
|---|---|---|
| YOLOv5 / v6 / v8 | `[1,K,6]`（score, class, x1, y1, x2, y2） | 框解码 + NMS 已在模型内完成，只需坐标还原 |
| YOLO26 (n/s/m) | 6 个原始检测头（stride 8/16/32 的 box + score） | host 端 sigmoid + 框解码 + 按类 NMS |

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源） |
| 2 | 选择模型和图片 |
| 3 | 前处理（letterbox 等比缩放 + 灰边填充） |
| 4 | RKNN 推理 |
| 5 | 后处理与可视化 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。


本 demo 会下载 YOLOv5/v6/v8 和 YOLO26 n/s/m（各 i8/fp16 两版），以及 COCO 80 类标签和示例图片。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("yolo-rknn3-rk1828", "yolov5/i8/yolov5.rknn", "models/yolo/yolov5/i8/yolov5.rknn"),
    ("yolo-rknn3-rk1828", "yolov5/i8/yolov5.weight", "models/yolo/yolov5/i8/yolov5.weight"),
    ("yolo-rknn3-rk1828", "yolov5/fp16/yolov5.rknn", "models/yolo/yolov5/fp16/yolov5.rknn"),
    ("yolo-rknn3-rk1828", "yolov5/fp16/yolov5.weight", "models/yolo/yolov5/fp16/yolov5.weight"),
    ("yolo-rknn3-rk1828", "yolov6/i8/yolov6.rknn", "models/yolo/yolov6/i8/yolov6.rknn"),
    ("yolo-rknn3-rk1828", "yolov6/i8/yolov6.weight", "models/yolo/yolov6/i8/yolov6.weight"),
    ("yolo-rknn3-rk1828", "yolov6/fp16/yolov6.rknn", "models/yolo/yolov6/fp16/yolov6.rknn"),
    ("yolo-rknn3-rk1828", "yolov6/fp16/yolov6.weight", "models/yolo/yolov6/fp16/yolov6.weight"),
    ("yolo-rknn3-rk1828", "yolov8/i8/yolov8.rknn", "models/yolo/yolov8/i8/yolov8.rknn"),
    ("yolo-rknn3-rk1828", "yolov8/i8/yolov8.weight", "models/yolo/yolov8/i8/yolov8.weight"),
    ("yolo-rknn3-rk1828", "yolov8/fp16/yolov8.rknn", "models/yolo/yolov8/fp16/yolov8.rknn"),
    ("yolo-rknn3-rk1828", "yolov8/fp16/yolov8.weight", "models/yolo/yolov8/fp16/yolov8.weight"),
    ("yolo-rknn3-rk1828", "yolo26n/i8/yolo26n.rknn", "models/yolo/yolo26n/i8/yolo26n.rknn"),
    ("yolo-rknn3-rk1828", "yolo26n/i8/yolo26n.weight", "models/yolo/yolo26n/i8/yolo26n.weight"),
    ("yolo-rknn3-rk1828", "yolo26n/fp16/yolo26n.rknn", "models/yolo/yolo26n/fp16/yolo26n.rknn"),
    ("yolo-rknn3-rk1828", "yolo26n/fp16/yolo26n.weight", "models/yolo/yolo26n/fp16/yolo26n.weight"),
    ("yolo-rknn3-rk1828", "yolo26s/i8/yolo26s.rknn", "models/yolo/yolo26s/i8/yolo26s.rknn"),
    ("yolo-rknn3-rk1828", "yolo26s/i8/yolo26s.weight", "models/yolo/yolo26s/i8/yolo26s.weight"),
    ("yolo-rknn3-rk1828", "yolo26s/fp16/yolo26s.rknn", "models/yolo/yolo26s/fp16/yolo26s.rknn"),
    ("yolo-rknn3-rk1828", "yolo26s/fp16/yolo26s.weight", "models/yolo/yolo26s/fp16/yolo26s.weight"),
    ("yolo-rknn3-rk1828", "yolo26m/i8/yolo26m.rknn", "models/yolo/yolo26m/i8/yolo26m.rknn"),
    ("yolo-rknn3-rk1828", "yolo26m/i8/yolo26m.weight", "models/yolo/yolo26m/i8/yolo26m.weight"),
    ("yolo-rknn3-rk1828", "yolo26m/fp16/yolo26m.rknn", "models/yolo/yolo26m/fp16/yolo26m.rknn"),
    ("yolo-rknn3-rk1828", "yolo26m/fp16/yolo26m.weight", "models/yolo/yolo26m/fp16/yolo26m.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/coco_80_labels_list.txt", "datasets/coco_80_labels_list.txt"),
    ("yolo26-segment-rknn3-rk1828", "model/bus.jpg", "datasets/bus.jpg"),
    ("mobilenet_v2-rknn3-rk1828", "dog_224x224.jpg", "datasets/dog_224x224.jpg"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 选择模型和图片

模型文件在 `models/yolo/<名称>/<量化>/` 下；图片可以换成自己的。

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(
    options=["yolov5", "yolov6", "yolov8", "yolo26n", "yolo26s", "yolo26m"],
    value="yolov8", description="模型:")
w_quant = widgets.Dropdown(options=["i8", "fp16"], description="量化:")
w_image = widgets.Dropdown(options=sorted(glob.glob("datasets/*.jpg")),
                           description="图片:", layout=widgets.Layout(width="60%"))
display(w_model, w_quant, w_image)

LABELS = open("datasets/coco_80_labels_list.txt").read().splitlines()
print(f"{len(LABELS)} 个 COCO 类别。选好后继续往下运行 ▼")

## 3. 前处理：letterbox

YOLO 要求 640×640 正方形输入。直接拉伸会改变物体长宽比，所以**等比缩放后灰边填充**，
并记下缩放比例和 padding —— 后处理时要用它们把框还原回原图坐标。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from vision_post import letterbox, to_input_array, setup_cjk_font

setup_cjk_font()
img = Image.open(w_image.value).convert("RGB")
img_lb, SCALE, PAD_X, PAD_Y = letterbox(img, 640)
x = to_input_array(img_lb)
print(f"原图 {img.size} -> 输入 {x.shape}, scale={SCALE:.3f}, pad=({PAD_X},{PAD_Y})")

# 前处理每个阶段都画出来：原图 -> letterbox -> 输入张量
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, im, title in [
    (axes[0], img, f"1. 原图 {img.size[0]}x{img.size[1]}"),
    (axes[1], img_lb, "2. letterbox 640x640\n(等比缩放 + 灰边填充)"),
    (axes[2], x[0], f"3. 模型输入 {x.shape}\nuint8 NHWC"),
]:
    ax.imshow(np.asarray(im))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. RKNN 推理

In [ ]:
import time
from rknn3 import Rknn3Model

model_path = f"models/yolo/{w_model.value}/{w_quant.value}/{w_model.value}.rknn"
model = Rknn3Model(model_path)
print(f"{len(model.output_attrs)} 个输出:")
for a in model.output_attrs:
    print("  ", a)

_ = model.infer([x])
t0 = time.time()
outputs = model.infer([x])
print(f"推理耗时: {(time.time()-t0)*1000:.1f} ms")

## 5. 后处理

- **YOLOv5/v6/v8**：输出已经是 NMS 后的 `(score, class, x1, y1, x2, y2)`，按置信度过滤即可
- **YOLO26**：6 个检测头（3 个 box 头 `[1,4,h,w]` + 3 个 score 头 `[1,80,h,w]`），
  在 host 端做 sigmoid、框解码和按类 NMS；i8 输出还要先反量化

In [ ]:
from vision_post import (decode_builtin_nms, decode_yolo26_heads,
                         unletterbox_boxes)

if w_model.value.startswith("yolo26"):
    boxes, classes, scores = decode_yolo26_heads(outputs, model.output_attrs,
                                                 conf_thresh=0.4)
else:
    boxes, classes, scores = decode_builtin_nms(outputs[0], conf_thresh=0.4)

boxes = unletterbox_boxes(boxes, SCALE, PAD_X, PAD_Y, *img.size)
print(f"检测到 {len(boxes)} 个目标:")
for b, c, s in zip(boxes, classes, scores):
    print(f"  {LABELS[c]:12s} {s:.2f}  box=({b[0]:.0f},{b[1]:.0f},{b[2]:.0f},{b[3]:.0f})")

## 6. 可视化

In [ ]:
from vision_post import draw_boxes

result = draw_boxes(img, boxes, classes, scores, LABELS)
model.close()
result

## 7. 多模型对比（i8 + fp16 全量）

同一张图，把 **6 个模型 × 2 种量化（i8 / fp16）共 12 组**全部跑一遍。
运行过程实时打印进度；全部完成后输出对比表格、耗时柱状图和检测效果拼图。
每个模型计时 3 次取最快（排除首次预热抖动）。

In [ ]:
import time
from rknn3 import Rknn3Model
from vision_post import (decode_builtin_nms, decode_yolo26_heads,
                         unletterbox_boxes)

ALL_MODELS = ["yolov5", "yolov6", "yolov8", "yolo26n", "yolo26s", "yolo26m"]
QUANTS = ["i8", "fp16"]

compare = []  # (model, quant, latency_ms, boxes, classes, scores)
total = len(ALL_MODELS) * len(QUANTS)
step = 0
for name in ALL_MODELS:
    for quant in QUANTS:
        step += 1
        print(f"[{step:2d}/{total}] 加载 {name} ({quant}) ...", flush=True)
        m = Rknn3Model(f"models/yolo/{name}/{quant}/{name}.rknn")
        m.infer([x])  # 预热
        best, outs = float("inf"), None
        for _ in range(3):
            t0 = time.time()
            outs = m.infer([x])
            best = min(best, (time.time() - t0) * 1000)
        if name.startswith("yolo26"):
            b, c, s = decode_yolo26_heads(outs, m.output_attrs, conf_thresh=0.4)
        else:
            b, c, s = decode_builtin_nms(outs[0], conf_thresh=0.4)
        b = unletterbox_boxes(b, SCALE, PAD_X, PAD_Y, *img.size)
        compare.append((name, quant, best, b, c, s))
        m.close()
        print(f"      -> {best:6.1f} ms, {len(b)} 个目标", flush=True)
print("\n全部完成 ✅", flush=True)

### 对比表格

In [ ]:
from IPython.display import HTML

rows = "".join(
    f"<tr><td>{n}</td><td>{q}</td>"
    f"<td>{lat:.1f}</td><td>{len(b)}</td>"
    f"<td>{', '.join(f'{LABELS[ci]}({si:.2f})' for ci, si in list(zip(c, s))[:5]) or '-'}</td></tr>"
    for n, q, lat, b, c, s in compare)
HTML("<table style='text-align:left'>"
     "<tr><th>模型</th><th>量化</th><th>耗时(ms)</th><th>目标数</th><th>Top-5 检测</th></tr>"
     + rows + "</table>")

### 耗时柱状图（i8 vs fp16）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
colors = {"i8": "#3b82f6", "fp16": "#f59e0b"}
xs = np.arange(len(ALL_MODELS))
width = 0.38
fig, ax = plt.subplots(figsize=(10, 4.5))
for k, q in enumerate(QUANTS):
    lats = [lat for n, qq, lat, *_ in compare if qq == q]
    cnts = [len(b) for n, qq, _, b, *_ in compare if qq == q]
    bars = ax.bar(xs + (k - 0.5) * width, lats, width, label=q, color=colors[q])
    for bar, lat, cnt in zip(bars, lats, cnts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{lat:.0f}ms\n{cnt}个", ha="center", va="bottom", fontsize=8)
ax.set_xticks(xs, ALL_MODELS)
ax.set_ylabel("推理耗时 (ms)")
ax.set_title("YOLO 多模型对比（同一张图，i8 vs fp16）")
ax.set_ylim(0, max(lat for _, _, lat, *_ in compare) * 1.3)
ax.legend()
plt.tight_layout()
plt.show()

### 检测效果拼图（上排 i8，下排 fp16）

In [ ]:
import matplotlib.pyplot as plt
from vision_post import draw_boxes

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for k, q in enumerate(QUANTS):
    row = [(n, lat, b, c, s) for n, qq, lat, b, c, s in compare if qq == q]
    for ax, (n, lat, b, c, s) in zip(axes[k], row):
        ax.imshow(draw_boxes(img, b, c, s, LABELS))
        ax.set_title(f"{n} ({q})  {lat:.0f}ms  {len(b)}个", fontsize=10)
        ax.axis("off")
plt.tight_layout()
plt.show()

## 小结

- YOLOv5/v6/v8 把解码和 NMS 做进了模型，host 端几乎零后处理；
  YOLO26 输出原始检测头换取部署灵活性，后处理在 host 完成（见 `vision_post.decode_yolo26_heads`）。
- 一般规律：i8 比 fp16 快，模型越大越慢但检测更全；
  官方参考 mAP@0.5:0.95: 26n 40.9 / 26s 48.6 / 26m 53.1（PyTorch FP32）。
- 模型命名说明：v5/v6/v8 是瑞芯微官方 SDK 的预转换模型，按家族名发布未带规格后缀；
  从权重体积看 yolov5 接近 **yolov5s**（fp16 16.2MB），并非 yolov5n；
  v6/v8 的 i8 权重文件反而比 fp16 大（i8 版本内嵌的 NMS 后处理头以 fp16 保存所致）。